# Instationäre Behälterleerung -- zweiter Lösungsweg: rein numerisch

**BIW3-10 Weiterführende Hydromechanik · Übung 2 · Aufgabe 5**

Auf dem Blatt wurde die Bewegungsgleichung hergeleitet (Volumenbilanz $+$ Torricelli) und $h(t)$ geschlossen gelöst. **Dieses Notebook geht einen zweiten Weg und braucht die hergeleitete Gleichung dafür nicht:**

- die **Physik wird direkt programmiert**: der momentane Abfluss nach Torricelli senkt den Spiegel (Kontinuität),
- **scipy** integriert die Bewegung in der Zeit,
- die hergeleitete Gleichung $h(t) = h_0\left(1-\frac{t}{T_E}\right)^2$ taucht **erst ganz am Ende** auf -- als Kontrolle der Numerik.

**Lernziele:** Physik als Funktion `f(t, y)` aufschreiben · mit `solve_ivp` lösen · die Instationarität in der Zeitreihe sehen · begründen, warum ohne Herleitung dieselbe Kurve entsteht.

## 1. Parameter (wie in Aufgabe 4 des Übungsblatts)

Zylinderspeicher mit scharfkantiger Bodenöffnung, kein Zulauf. Alle Werte in **SI-Einheiten**.

In [ ]:
# ======================================================================
# 1. PARAMETER DER AUFGABE 4 (Übungsblatt) -- alle in SI-Einheiten!
# ======================================================================
# Jede Zeile: Name = Wert   # Einheit | Bedeutung

g     = 9.81    # m/s^2  | Erdbeschleunigung
D     = 2.0     # m      | Behälter-Innendurchmesser (Zylinder)
d_o   = 0.08    # m      | Durchmesser der Bodenöffnung (Blatt: 80 mm -> in Meter)
alpha = 0.62    # -      | Ausflusszahl (Folie 16): Strahlverengung + Verluste
h0    = 3.6     # m      | Anfangswasserstand über der Öffnung -- unsere Unbekannte h(t)

In [ ]:
# ======================================================================
# 2. FLÄCHEN - werden aus den Parametern berechnet
# ======================================================================
import numpy as np                 # Numerik-Bibliothek (sqrt, pi, Arrays, ...)
import matplotlib.pyplot as plt    # Plot-Bibliothek

A   = np.pi * D**2 / 4    # m^2 | Behälterquerschnitt (Kreisfläche)
A_o = np.pi * d_o**2 / 4  # m^2 | Öffnungsquerschnitt

# --- Vorschau auf die Kontinuitäts-Argumentation (Aufgabe 4a): ---------
#   Derselbe Volumenstrom Q fließt durch Spiegelfläche UND Öffnung:
#       Q = v_Spiegel * A = v_Ausfluss * A_o
#   ->  v_Spiegel = (A_o/A) * v_Ausfluss  -- hier nur 0,16 % !
#   Deshalb darf die Strömung im Behälter als vernachlässigbar gelten.
print(f"A   = {A:.4f} m^2")
print(f"A_o = {A_o*1e3:.3f} e-3 m^2")
print(f"A_o/A = {A_o/A*100:.2f} %  -> v_Spiegel ist winzig gegen v_Ausfluss")

## 2. Die Physik als Funktion (Aufgabe 5a)

**Ohne jede hergeleitete Formel** -- nur zwei Sätze aus der Vorlesung:

1. **Torricelli** (Folie 16): der Abfluss hängt vom *momentanen* Wasserstand ab,
   $$Q_{\mathrm{ab}} = \alpha\, A_{\mathrm{o}}\,\sqrt{2 g h}$$
2. **Kontinuität** (Volumenbilanz): der Abfluss senkt den Spiegel um $Q_{\mathrm{ab}}/A$ pro Zeit,
   $$\frac{\mathrm{d}h}{\mathrm{d}t} = -\frac{Q_{\mathrm{ab}}}{A}$$

Diese zwei Zeilen sind die **rechte Seite** $f(t,h)$ -- mehr braucht `scipy` nicht. Jede Auswertung ist eine Momentaufnahme (quasi-stationär, Folie 26).

In [ ]:
# ======================================================================
# 3. DIE PHYSIK ALS FUNKTION   (Aufgabe 5a -- TODO 1 + TODO 2)
# ======================================================================
def rhs(t, y):
    """Rechte Seite der Bewegung für scipy.integrate.solve_ivp.

    solve_ivp übergibt:
        t : aktuelle Zeit [s]   (hier unbenutzt -- die Physik hängt nur von h ab)
        y : Zustandsvektor      (y[0] = Wasserstand h [m])

    Rückgabe:
        dh/dt = -Q_ab / A   [m/s]
    """
    # Erst wenn TODO 1 + 2 unten umgesetzt sind, diese Zeile LÖSCHEN:
    raise NotImplementedError("Aufgabe 5a: Erst TODO 1 + 2 in rhs() umsetzen!")

    h = y[0]                        # Wasserstand aus dem Zustandsvektor

    # ---------------- TODO 1: momentaner Abfluss (Torricelli) ---------
    #   Q_ab = alpha * A_o * sqrt(2 * g * h)      [m^3/s]
    #   Tipp: np.sqrt(...) und max(h, 0.0) -- gegen Wurzeln aus
    #   negativen Zahlen (kann am Ende der Rechnung passieren).
    Q_ab = ...

    # ---------------- TODO 2: Kontinuität -----------------------------
    #   Der Abfluss senkt den Spiegel um Q_ab/A pro Zeit:
    #   dh/dt = -Q_ab / A
    dhdt = ...

    return dhdt

## 3. Zeitintegration mit scipy (Aufgabe 5b)

`scipy.integrate.solve_ivp` integriert $\dd y/\dd t = f(t,y)$ automatisch (Runge-Kutta 4/5, **adaptive Schrittweite** -- groß, wo sich wenig tut; klein, wo es steil wird). Wir übergeben:

- die **Physik-Funktion** `rhs` (von oben),
- den **Zeitbereich** `(0, 3600)` -- eine Stunde, großzügig: ohne die hergeleitete Formel kennen wir die Dauer noch nicht, das Event stoppt automatisch,
- den **Anfangszustand** `[h0]` (Vektor! später wären mehrere Unbekannte möglich),
- ein **Ereignis (Event)**: "Behälter leer" bei $h = 0$ -- dort stoppt die Rechnung,
- Genauigkeit: `rtol=1e-8`, `atol=1e-10`.

In [ ]:
# ======================================================================
# 4. ZEITINTEGRATION   (Aufgabe 5b -- TODO 3, 4, 5)
# ======================================================================
from scipy.integrate import solve_ivp

# ---------------- TODO 3: Event "Behälter leer" -----------------------
#   Eine Event-Funktion gibt die Größe zurück, deren Nulldurchgang
#   interessiert: hier der Wasserstand y[0]. Sie wird genau dann null,
#   wenn h = 0 ist (Behälter bis zur Öffnung entleert).
def event_leer(t, y):
    return ...        # <-- TODO 3

event_leer.terminal = True      # Rechnung BEIM Event anhalten
event_leer.direction = -1       # nur Übergang von + nach - zählen

# ---------------- TODO 4: Zeitintegration starten ---------------------
#   Vorlage (ausfüllen):
#   sol = solve_ivp(rhs, t_span, y0, events=event_leer, rtol=1e-8, atol=1e-10)
#   mit  t_span = (0, 3600)  [1 Stunde -- großzügig, das Event stoppt rechtzeitig]  und  y0 = [h0]
t_span = (0, 3600)
y0 = [h0]
sol = None       # <-- TODO 4: hier das Ergebnis von solve_ivp speichern

# ---------------- TODO 5: Auslaufzeit ablesen -------------------------
#   Der Event-Zeitpunkt steht in sol.t_events[0][0].
T_E_num = None   # <-- TODO 5

if T_E_num is None:
    raise NotImplementedError("Aufgabe 5b: Erst TODO 3, 4 + 5 umsetzen!")

print(f"T_E,num = {T_E_num:.2f} s = {T_E_num/60:.2f} min")

## 4. Ergebnis: die Zeitreihe (Aufgabe 5c)

Die gelöste Zeitreihe $h(t)$ ist die Instationarität zum Anfassen: am festen Ort (dem Behälter) ändert sich der Zustand ständig -- $\partial h/\partial t \neq 0$. Der Abfluss fällt mit: $Q(t) = \alpha A_{\mathrm{o}}\sqrt{2g\,h(t)}$.

In [ ]:
# ======================================================================
# 5. PLOT: h(t) mit Sensor-Wasserstand   (Aufgabe 5c -- TODO 6)
# ======================================================================
plt.figure(figsize=(8, 4))
plt.plot(sol.t, sol.y[0], 'b-', linewidth=2, label='numerisch: h(t)')

# ------------- TODO 6: Sensor-Wasserstand einzeichnen -----------------
#   Horizontale Linie bei h = 1.0 m (Sensor aus Aufgabe 4e):
#   plt.axhline(1.0, color='orange', linestyle='--', label='Sensor (1,0 m)')

plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Instationäre Behälterleerung -- rein numerisch gerechnet')
plt.show()

# Beobachtung: h fällt anfangs SCHNELL (viel Wasser drückt -> großer Abfluss),
# am Ende kaum noch -- die Kurve ist eine Parabel. Aber Moment:
# die Parabel-FORM war ja nirgends vorgegeben worden ... oder doch?

## 5. Erst jetzt: Vergleich mit der hergeleiteten Gleichung

**Jetzt** holen wir die aus der Übung hergeleitete Gleichung -- sie dient nur noch als **Kontrolle** der Numerik:

$$h(t) = h_0\left(1-\frac{t}{T_E}\right)^{2}, \qquad T_E = \frac{2A\sqrt{h_0}}{\alpha A_{\mathrm{o}}\sqrt{2g}}$$

Wenn der numerische Weg korrekt ist, muss er auf derselben Kurve landen -- **ohne dass die Formel irgendwo in den Code eingegangen ist**.

In [ ]:
# ======================================================================
# 6. VERGLEICH MIT DER HERGELEITETEN GLEICHUNG   (Kontrolle)
# ======================================================================
# Die Formel aus der Übung -- erst HIER kommt sie vor:
T_E_exakt = 2 * A * np.sqrt(h0) / (alpha * A_o * np.sqrt(2*g))   # s

def h_hergeleitet(t):
    """Aus der Übung: h(t) = h0*(1 - t/T_E)^2 -- Kontrollkurve."""
    return h0 * (1 - t/T_E_exakt)**2

t_fein = np.linspace(0, T_E_exakt, 200)

plt.figure(figsize=(8, 4))
plt.plot(t_fein, h_hergeleitet(t_fein), 'r--', linewidth=1.5, label='hergeleitete Gleichung (exakt)')
plt.plot(sol.t, sol.y[0], 'b-', linewidth=2, label='numerisch (ohne Herleitung)')
plt.axhline(1.0, color='orange', linestyle=':', label='Sensor (1,0 m)')
plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Zweiter Lösungsweg trifft den ersten')
plt.show()

print(f"T_E,num   = {T_E_num:.2f} s")
print(f"T_E,exakt = {T_E_exakt:.2f} s")
print(f"Abweichung = {(T_E_num - T_E_exakt)/T_E_exakt*100:+.4f} %")

# Zusatz: Anfangs- und Endverhalten der Absenkung (Instationarität!)
print(f"dh/dt(0)      = {rhs(0, [h0])*1000:+.2f} mm/s  (maximale Absenkung)")
print(f"dh/dt(0.5T_E) = {rhs(T_E_exakt/2, [h0/4])*1000:+.2f} mm/s  (halber Abfluss, da Q ~ sqrt(h))")

## 6. Interpretation (Aufgabe 5d) -- in eigenen Worten beantworten

1. **Warum kommt trotz Verzicht auf die hergeleitete Gleichung dieselbe Parabel heraus?** *(Welche Physik steckt in `rhs` -- und welche in der Herleitung auf dem Blatt?)*
2. **Was macht die Rechnung instationär?** Welche Größe ändert sich am festen Ort mit der Zeit? *($\partial h/\partial t \neq 0$ -- und $Q(t)$ fällt.)*
3. **Warum ist die Absenkung am Anfang am schnellsten?** *(Wie hängt $Q_{\mathrm{ab}}$ von $h$ ab?)*
4. **Wie müsste `rhs` aussehen, damit $h$ linear fällt?** *(Pumpe mit konstantem Abfluss -- Aufgabe 3a.)*

*Lösungshinweise stehen im LOESUNG-Notebook -- erst selbst überlegen!*